# 04_tracks

DML: bronze_tracks — Raw track metadata.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("tracks", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.explode("tracks").alias("t"))
    .select(
        F.col("t.id").alias("track_id"),
        F.col("t.name").alias("track_name"),
        F.col("t.duration_ms").alias("duration_ms"),
        F.col("t.popularity").alias("popularity"),
        F.col("t.explicit").alias("explicit"),
        F.col("t.artists.id").alias("artist_ids"),
        F.col("t.artists.name").alias("artist_names"),
        F.col("t.album.id").alias("album_id"),
        F.col("t.album.name").alias("album_name"),
        F.col("t.album.release_date").alias("album_release_date"),
        F.to_json(F.col("t")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_tracks")
print(f"bronze_tracks: {bronze.count()} rows written")